In [1]:
import pandas as pd

input_df = pd.read_excel("./input.xlsx", sheet_name="links")
input_df.drop(columns=["Unnamed: 9"])
input_df.head()

,program_id_,University,Faculty,Program and link,ON Admission Requirement,IB Admission Requirement,BC Admission Requirement,AP Admission Requirement,Language Requirement,Unnamed: 9
0,1,Algoma University,Science,Biology,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,NaN
1,2,Algoma University,Science,Environmental Science,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,NaN
2,3,Algoma University,Science,Computer Science,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,NaN
3,4,Algoma University,Arts,English,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,NaN
4,5,Algoma University,Arts,History,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,https://algomau.ca/students/international-stud...,https://algomau.ca/admissions/admissions-requi...,NaN


In [2]:
# sliced_input_df = input_df[200:210]
sliced_input_df = input_df[200:]

In [3]:
input_dict = sliced_input_df.to_dict(orient="index")

In [4]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
program_dict = {}
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
result_dict = {}

def process_row(row):
    def assign(career_direction):
        uni_name = row["University"]
        id_ = row["program_id_"]
        key = f"({id_}, {uni_name}, {program_name})"
        row_json = {
            "program_id_": id_,
            "university_name": uni_name,
            "program_name": program_name,
            "career_direction": career_direction,
        }
        print(key)
        result_dict[key] = row_json
    program_name = row["Program and link"]
    if program_name in program_dict:
        assign(program_dict[program_name])
        return program_dict[program_name]
    prompt = f"""
    I am a student studying {program_name}. 
    Please suggest 5-8 most important potential career directions for me, based on the skills and knowledge typically gained in this program. 
    Provide a list of specific career name (like options or majors) without any description. Separate each path by newline char.
    ## example output: 
        Accounting\nBusiness Technology Management\nEntrepreneurship\nFinance\nGeneral Business Management\nGlobal Supply Chain and Logistics Management\nMarketing\nOperations and Logistics\nOrganizational Behaviour and Human Resources\nReal Estate
    ## format:
        Don't place any leading white space
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
        n=1,
        stop=None,
        temperature=0
    )
    # pt_client.chat.completions.create(messages=messages, model=model, temperature=temperature)
    career_direction = response.choices[0].message.content
    career_direction = career_direction.replace("  \n", "\n").strip()
    program_dict[program_name] = career_direction
    # print(career_direction)
    assign(career_direction)
    return career_direction


In [6]:
import concurrent.futures



# Define the number of threads in the pool
max_threads = 10  # Adjust this based on your system capabilities

# Use ThreadPoolExecutor to manage the threads
with concurrent.futures.ThreadPoolExecutor(max_workers=max_threads) as executor:
    # Submit each task to the thread pool
    futures = {executor.submit(process_row, value): key for key, value in input_dict.items()}

    # Optional: Process the results as they complete
    for future in concurrent.futures.as_completed(futures):
        key = futures[future]
        try:
            result = future.result()
            # Optionally process the result
        except Exception as e:
            print(f"Row processing failed for {key}: {e}")


(209, Nipissing University
, Business)
(203, Nipissing University
, Biology)
(207, Nipissing University
, Adult Education)
(210, Nipissing University
, Criminal Justice)
(206, Nipissing University
, Special Education)
(205, Nipissing University
, Early Childhood Education)
(208, Nipissing University
, Educational Leadership)
(202, Nipissing University
, English Studies)
(204, Nipissing University
, Education)
(214, Nipissing University
, Business Administration)
(220, Nipissing University
, Special Education)
(221, Nipissing University
, Educational Leadership)
(201, Nipissing University
, History)
(213, Nipissing University
, Physical and Health Education)
(224, University of Ottawa, Biology)
(211, Nipissing University
, Nursing)
(217, Nipissing University
, Accounting)
(216, Nipissing University
, Marketing)
(212, Nipissing University
, Social Work)
(215, Nipissing University
, Finance)
(218, Nipissing University
, Human Resources)
(231, University of Ottawa, History)
(219, Nipissing

In [7]:
print(result_dict)

{'(209, Nipissing University\n, Business)': {'program_id_': 209, 'university_name': 'Nipissing University\n', 'program_name': 'Business', 'career_direction': 'Accounting\nBusiness Technology Management\nEntrepreneurship\nFinance\nGeneral Business Management\nGlobal Supply Chain and Logistics Management\nMarketing\nOperations and Logistics\nOrganizational Behaviour and Human Resources\nReal Estate\nBusiness Analytics\nConsulting\nInternational Business\nSales Management\nCorporate Strategy\nProject Management\nRisk Management\nPublic Relations\nInvestment Banking\nHuman Resource Management'}, '(203, Nipissing University\n, Biology)': {'program_id_': 203, 'university_name': 'Nipissing University\n', 'program_name': 'Biology', 'career_direction': 'Biotechnology\nMicrobiology\nGenetics\nEnvironmental Science\nMarine Biology\nEcology\nZoology\nBotany\nBiomedical Research\nPharmaceutical Sales\nForensic Science\nBioinformatics\nWildlife Conservation\nAgricultural Science\nImmunology\nPublic 

In [8]:
print(len(result_dict))

514


In [9]:
df = pd.DataFrame.from_dict(result_dict, orient="index").reset_index()
df.head(20)

,index,program_id_,university_name,program_name,career_direction
0,"(209, Nipissing University\n, Business)",209,Nipissing University\n,Business,Accounting\nBusiness Technology Management\nEn...
1,"(203, Nipissing University\n, Biology)",203,Nipissing University\n,Biology,Biotechnology\nMicrobiology\nGenetics\nEnviron...
2,"(207, Nipissing University\n, Adult Education)",207,Nipissing University\n,Adult Education,Adult Education Instructor\nCorporate Trainer\...
3,"(210, Nipissing University\n, Criminal Justice)",210,Nipissing University\n,Criminal Justice,Corrections Officer\nProbation Officer\nParole...
4,"(206, Nipissing University\n, Special Education)",206,Nipissing University\n,Special Education,Special Education Teacher\nResource Room Teach...
5,"(205, Nipissing University\n, Early Childhood ...",205,Nipissing University\n,Early Childhood Education,Preschool Teacher\nKindergarten Teacher\nChild...
6,"(208, Nipissing University\n, Educational Lead...",208,Nipissing University\n,Educational Leadership,School Principal\nAssistant Principal\nDistric...
7,"(202, Nipissing University\n, English Studies)",202,Nipissing University\n,English Studies,Copywriter\nEditor\nContent Strategist\nTechni...
8,"(204, Nipissing University\n, Education)",204,Nipissing University\n,Education,Teacher\nSchool Administrator\nCurriculum Deve...
9,"(214, Nipissing University\n, Business Adminis...",214,Nipissing University\n,Business Administration,Accounting\nBusiness Technology Management\nEn...


In [10]:
df.to_csv("./program_output_rerun.csv", index=False, encoding="utf-8")